# Interactive Plot

In [2]:
# %matplotlib widget
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.widgets import Slider, RadioButtons
from mpl_toolkits.mplot3d import Axes3D

# --- 1. DATA LOADING ---

class_name = '(T)UD_T35_R1'
modal_parameters = np.load(f"Modal Parameter Extracted NPY3\\Modalparameters_{class_name}.npy")

num_modes = modal_parameters.shape[0]
print(f"Available modes: 1 to {num_modes}")

nat_frequencies = np.real(modal_parameters[: , 0])

# --- 2. GEOMETRY SETUP ---
points = np.array([
    [-0.184, 0.08, 0],
    [-0.184, 0, 0],
    [-0.184, -0.08, 0],
    [0, 0.080, 0],
    [0, 0, 0],
    [0, -0.080, 0],
    [0.184, 0.08, 0],
    [0.184, 0, 0],
    [0.184, -0.08, 0]
    ])

# --- 3. PRE-PROCESS ALL MODES ---
mode_shapes = modal_parameters[: , 2:]

# Auto-scale mode shape to 0.1 max magnitude
max_values = np.max(np.abs(mode_shapes), axis=1)
mode_shapes = mode_shapes / max_values[:, np.newaxis] * 0.1

# --- 4. CONNECTIVITY DEFINITION ---
# 1. Connect 1-4-7-8-9-6-3-2-1  (Indices: 0, 3, 6, 7, 8, 5, 2, 1, 0)
# 2. Connect 4-5-6              (Indices: 3, 4, 5)
# 3. Connect 2-5-8              (Indices: 1, 4, 7)
connectivity = [
    [0, 3, 6, 7, 8, 5, 2, 1, 0],    # Base loop
    [3, 4, 5],                      # Horizontal line
    [1, 4, 7],                      # Vertical line
    ]

# --- 5. SETUP PLOT & LAYOUT ---
fig = plt.figure(figsize=(12, 8))
plt.subplots_adjust(left=0.25, bottom=0.25)
ax = fig.add_subplot(111, projection='3d')

# Lists to store the plot objects for each segment
lines_ref = []  # The static "ghost" lines
lines_def = []  # The moving "deformed" lines

# Create the line objects based on connectivity
for path in connectivity:
    # Get the coordinates for this specific path
    # path is a list of indices, e.g., [0, 1, 4, 3, 0]
    pts = points[path]
    
    # 1. Plot Reference (Static, Gray dashed)
    l_ref, = ax.plot(pts[:,0], pts[:,1], pts[:,2], 'k--', alpha=0.3)
    lines_ref.append(l_ref)
    
    # 2. Plot Deformed (Dynamic, Blue solid)
    # Initialize with empty data, will be filled in update()
    l_def, = ax.plot([], [], [], 'b-o', linewidth=2, markersize=4)
    lines_def.append(l_def)

# Initialize Frequency Text Display (Top Left of Plot)
# transform=ax.transAxes ensures 0,0 is bottom-left and 1,1 is top-right of the plot area
freq_text = ax.text2D(0.05, 0.95, "", transform=ax.transAxes, fontsize=10, color='black', fontweight='bold')

# Set Axis Limits
max_range = 0.2
mid_x, mid_y, mid_z = np.mean(points, axis=0)
ax.set_xlim(mid_x - max_range, mid_x + max_range)
ax.set_ylim(mid_y - max_range, mid_y + max_range)
ax.set_zlim(mid_z - max_range, mid_z + max_range)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
# ax.set_title("Mode Shape")

# --- 6. WIDGETS ---

# A. Sliders
ax_scale = plt.axes([0.35, 0.1, 0.55, 0.03])
ax_speed = plt.axes([0.35, 0.05, 0.55, 0.03])

s_scale = Slider(ax_scale, 'Amplitude', 0.0, 5.0, valinit=1.0)
s_speed = Slider(ax_speed, 'Speed', 0.0, 0.5, valinit=0.1)

# B. Radio Buttons
labels = [f"Mode {i+1}" for i in range(num_modes)]
ax_radio = plt.axes([0.02, 0.4, 0.15, 0.4]) 
radio = RadioButtons(ax_radio, labels, active=0)



# --- 4. ANIMATION UPDATE LOGIC ---
current_t = 0.0

def update(frame):
    global current_t
    
    # Get slider values
    amp_factor = s_scale.val
    speed_step = s_speed.val
    
    # Get Selected Mode Index
    current_label = radio.value_selected
    mode_index = labels.index(current_label)
    
    # Update Frequency Text
    current_freq = nat_frequencies[mode_index]
    freq_text.set_text(f"f\u2099: {current_freq:.2f} Hz")
    
    # Increment time
    current_t += speed_step
    
    # 1. Calculate FULL Displacement for all 9 points first
    # This keeps the math efficient (only done once per frame)
    # Assuming Z-direction deformation here
    mode_shape = mode_shapes[mode_index]
    full_disp_z = np.real(mode_shape * np.exp(1j * current_t)) * amp_factor
    
    # Create the full array of deformed points
    deformed_points = points.copy()
    deformed_points[:, 2] += full_disp_z 
    # NOTE: If you have X/Y deformation, add it here:
    # deformed_points[:, 0] += full_disp_x
    # deformed_points[:, 1] += full_disp_y
    
    # 2. Update each line segment individually
    # zip() lets us loop over the plot object (line) and the index path (path) together
    for line, path in zip(lines_def, connectivity):
        # Extract the specific points for this path from the deformed set
        path_pts = deformed_points[path]
        
        line.set_data(path_pts[:, 0], path_pts[:, 1])
        line.set_3d_properties(path_pts[:, 2])
    
    return lines_def

# Start Animation
ani = animation.FuncAnimation(fig, update, interval=20, blit=False)

plt.show()

Available modes: 1 to 3


C:\Users\ycwon\AppData\Local\Temp\ipykernel_12096\1813543598.py:150: UserWarning: frames=None which we can infer the length of, did not pass an explicit *save_count* and passed cache_frame_data=True.  To avoid a possibly unbounded cache, frame data caching has been disabled. To suppress this warning either pass `cache_frame_data=False` or `save_count=MAX_FRAMES`.
  ani = animation.FuncAnimation(fig, update, interval=20, blit=False)


# Save to GIF

In [5]:
def create_connected_mode_shape_animation(coords, mode_vector, filename='connected_mode.gif', scale=1.0):
    """
    Animates a 3D mode shape with specific line connectivity.
    """
    
    # 1. Prepare Data
    coords = np.array(coords)
    mode_vector = np.array(mode_vector).flatten()
    
    # Define Connectivity (Indices are 0-based)
    connectivity = [
        [0, 1, 4, 3, 0],
        [1, 2, 3],
        [0, 2, 4],
        [5, 1, 7, 4, 8, 3, 6, 0, 5]
    ]
    
    # 2. Setup Figure
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Calculate Plot Bounds
    max_range = np.array([
        coords[:, 0].max()-coords[:, 0].min(), 
        coords[:, 1].max()-coords[:, 1].min(), 
        coords[:, 2].max()-coords[:, 2].min()
    ]).max() / 2.0
    mid_x = np.mean(coords[:, 0])
    mid_y = np.mean(coords[:, 1])
    mid_z = np.mean(coords[:, 2])
    
    # 3. Initialize Plot Elements
    lines_ref = [] # Static (Undeformed)
    lines_anim = [] # Animated (Deformed)
    
    for path in connectivity:
        # Plot Static Reference (Gray Dashed)
        pts = coords[path]
        ax.plot(pts[:,0], pts[:,1], pts[:,2], 'k--', alpha=0.3)
        
        # Initialize Animated Lines (Blue)
        line, = ax.plot([], [], [], 'b-', linewidth=2, marker='o', markersize=6)
        lines_anim.append(line)

    # 4. Animation Functions
    frames = 60
    fps = 20
    
    def init():
        ax.set_xlim(mid_x - max_range, mid_x + max_range)
        ax.set_ylim(mid_y - max_range, mid_y + max_range)
        ax.set_zlim(mid_z - max_range, mid_z + max_range)
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        return lines_anim

    def update(frame):
        # Current time in the cycle (0 to 2pi)
        t = (frame / frames) * 2 * np.pi
        
        # Calculate Displacement for ALL points
        # u(t) = Real(phi * e^(iwt))
        disp_z = np.real(mode_vector * np.exp(1j * t)) * scale
        
        # Create full deformed coordinate set
        # (Assuming Z-direction deformation only)
        deformed_coords = coords.copy()
        deformed_coords[:, 2] += disp_z
        
        # Update each line segment based on its specific path
        for line, path in zip(lines_anim, connectivity):
            path_pts = deformed_coords[path]
            line.set_data(path_pts[:, 0], path_pts[:, 1])
            line.set_3d_properties(path_pts[:, 2])
            
        return lines_anim

    # 5. Generate and Save
    ani = animation.FuncAnimation(fig, update, frames=frames, init_func=init, blit=False)
    
    print(f"Saving animation to {filename}...")
    if filename.endswith('.gif'):
        ani.save(filename, writer='pillow', fps=fps)
    elif filename.endswith('.mp4'):
        ani.save(filename, writer='ffmpeg', fps=fps)
    print("Done.")

In [6]:
points = np.array([
    [-0.184, 0, 0],
    [0, 0.080, 0],
    [0, 0, 0],
    [0, -0.080, 0],
    [0.184, 0, 0]
])

# Mode Shape
mode_shape = modal_parameters[mode - 1, 2:]

points_new = np.array([
    [-0.184, 0.080, 0],
    [-0.184, -0.080, 0],
    [0.184, 0.080, 0],
    [0.184, -0.080, 0]
])

rbf = RBFInterpolator(points, mode_shape, kernel='linear')
mode_shape_new = rbf(points_new)

points = np.vstack((points, points_new))
mode_shape = np.append(mode_shape, mode_shape_new)

# Auto-scale mode shape to 0.1 max magnitude
mode_shape = mode_shape / np.max(np.abs(mode_shape)) * 0.1

# Create the file
create_connected_mode_shape_animation(points, mode_shape, filename=f'../data/processed/plate_new/mode shape/modeshape_{class_name}_mode{mode}.gif', scale=1)

NameError: name 'mode' is not defined